# 多产品竞品情报摘要器：网页抓取 + LLM

## 练习目标（理念）

用 **Selenium** 打开真实浏览器抓取产品页正文，再分别交给：

- 本地 **Ollama（llama3.2）**：通过 `subprocess` 调 `ollama run`
- 云端 **OpenAI（gpt-4o-mini）**：通过 Chat Completions API

对同一批产品文本做摘要，并排打印，对比两套后端的风格与可用性。

## 和本课 Day 1 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| `messages`（system / user） | `system_prompt` + 产品正文 |
| Chat Completions API | `openai.chat.completions.create(...)` |
| 网页抓取 | Selenium 无头 Chrome 取 `body.text` |
| 本地模型 | `ollama run llama3.2`（子进程） |

## 怎么跑

1. 准备 `.env`：`OPENAI_API_KEY`（以 `sk-proj-` 开头）
2. 安装一次依赖格（selenium / bs4 / requests）；本机需有 Chrome + chromedriver
3. 确保 Ollama 已启动且已 `ollama pull llama3.2`
4. 从上到下依次运行；最后一格会对每个产品先 Ollama 再 OpenAI 打印摘要


In [ ]:
# ========== 导入 + 加载并校验 OpenAI API Key ==========

# 导入标准库 os：读环境变量（Environment Variables），例如 OPENAI_API_KEY
import os
# 导入 requests：本 notebook 主路径用 Selenium，但依赖安装格也装了 requests，保留导入以匹配原逻辑
import requests
# 从 dotenv 导入 load_dotenv：把 .env 文件里的密钥读进环境变量，避免把密钥写进代码
from dotenv import load_dotenv
# 从 openai 导入 OpenAI 客户端类：调用云端 Chat Completions API
from openai import OpenAI

# 从 .env 加载环境变量；override=True：已存在的环境变量也用 .env 覆盖
load_dotenv(override=True)
# 取出 API Key 字符串（没有则为 None）
api_key = os.getenv('OPENAI_API_KEY')

# 检查 Key：分层提示常见配置错误（缺 key / 前缀不对 / 首尾空白）
# 注意：下面 print 文案保持英文原样（原 notebook 诊断字符串，不翻译以免改「给人看的固定话术」一致性；且属错误提示原文）

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")


In [ ]:
# ========== system 提示：定「怎么总结产品信息」的角色 ==========
# 定义 system 提示——可自行实验，例如改成用西班牙语 Markdown 回复
# 字符串保持英文：这是发给模型的指令，改译会改变回答风格/行为
system_prompt = "Summarize the following product information for comparison."


In [ ]:

# 安装依赖（只需运行一次）
# shell magic：!pip 在当前 Kernel 环境里安装包；selenium=浏览器自动化，bs4=HTML 解析，requests=HTTP
!pip install selenium bs4 requests


In [ ]:
# ========== 创建默认 OpenAI 客户端（读环境变量里的 OPENAI_API_KEY） ==========
openai = OpenAI()


In [ ]:
# ========== 路径 A：用 OpenAI Chat Completions 做产品摘要 ==========
# text：抓下来的产品页正文；model 默认 gpt-4o-mini（便宜、够用）
def summarize_with_openai(text, model="gpt-4o-mini"):
    # create：同步一次完整生成；temperature=0.7 略提高多样性（原参数保持）
    response = openai.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": text}
        ],
        temperature=0.7
    )
    # 取出第一条候选回复的正文
    return response.choices[0].message.content


In [ ]:

# ========== Selenium 无头浏览器：渲染 JS 后再取页面可见文本 ==========
# Selenium 无头配置
# webdriver：驱动本机 Chrome；Options：启动参数；By：元素定位策略；time：简单等待页面加载
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
import time

def scrape_text_from_url(url):
    # Options：Chrome 启动配置对象
    options = Options()
    # --headless=new：无头模式（不弹窗），适合服务器/笔记本批量抓取
    options.add_argument("--headless=new")
    # 按 options 启动 Chrome 实例
    driver = webdriver.Chrome(options=options)
    # 导航到目标 URL（会执行页面 JS）
    driver.get(url)
    # 粗暴等待 3 秒：给动态内容时间渲染（生产环境更常用显式等待）
    time.sleep(3)
    
    # 可按网站调整选择器：这里取整个 <body> 的可见文本
    body = driver.find_element(By.TAG_NAME, 'body')
    text = body.text
    # 关闭浏览器，释放进程
    driver.quit()
    # strip：去掉首尾空白后返回
    return text.strip()


In [ ]:

# ========== 路径 B：用本地 Ollama（llama3.2）经子进程做摘要 ==========
# 用 Ollama（本地 llama3）做 LLM 提示
# subprocess：在操作系统层面启动 `ollama` CLI，把 prompt 从 stdin 喂进去
import subprocess

def summarize_with_ollama(text):
    # 拼出英文 prompt：指示模型总结产品描述；正文插在中间
    prompt = f"Summarize the following product description:\n\n{text}\n\nSummary:"
    try:
        # 调试打印：进入 Ollama 调用分支（原文保留）
        print("inside ollama")
        # run：阻塞直到子进程结束；capture_output 收集 stdout/stderr；check=True 非零退出码抛异常
        result = subprocess.run(
            ["ollama", "run", "llama3.2"],
            input=prompt,
            capture_output=True, text=True, check=True, encoding="utf-8"
        )
        # 调试打印：拿到结果（原文保留）
        print("git result")
        # stdout：模型打印到标准输出的摘要文本
        return result.stdout.strip()
    except subprocess.CalledProcessError as e:
        # 失败时把 stderr 拼进错误字符串返回（不抛到外层，方便并排对比继续跑）
        return f"Error running ollama: {e.stderr}"


In [ ]:

# ========== 批量抓取：多个产品 URL → 名称到正文的字典 ==========
# 🔁 Analyze multiple product URLs and summarize
# 产品展示名 → 产品页 URL（印度区 apple/samsung 链接，原样保留）
product_urls = {
    "iPhone 15 Pro": "https://www.apple.com/in/iphone-15-pro/",
    "Samsung S24 Ultra": "https://www.samsung.com/in/smartphones/galaxy-s24-ultra/",
}

# 空字典：稍后填入「产品名 → 抓取文本」
product_texts = {}

# 逐个产品：打印进度 → Selenium 抓正文 → 存入字典
for name, url in product_urls.items():
    print(f"Scraping {name} ...")
    product_texts[name] = scrape_text_from_url(url)


In [ ]:

# ========== 并排对比：同一产品文本，先 Ollama 再 OpenAI ==========
# 📄 Display side-by-side summaries
for name, text in product_texts.items():
    # 本地路径：子进程调 ollama run
    print(f"\n🔹 {name} Summary with Ollama:")
    print(summarize_with_ollama(text))

    # 云端路径：Chat Completions
    print(f"\n🔹 {name} Summary with OpenAI GPT:")
    print(summarize_with_openai(text))
    # 分隔线：方便在长输出里肉眼分段
    print("="*100)
